# PyTorch Fundamentals — The Definitive Reference
**HackerRank Prep — Theme 0.6**

## TL;DR
PyTorch is the backbone of virtually every modern deep learning model used in structural biology, protein language models, and ML research. This notebook is the **shared foundation** for:
- Notebook 05 (Deep Learning)
- Notebook 06 (Graph Neural Networks)
- Notebook 07 (AlphaFold3 & protein structure)
- Notebook 10 (Fine-tuning transformers — OpenFOLD3 capstone)

Every concept introduced here will be used directly in those notebooks. Master this first.

---

## Learning Objectives
- [ ] Create, manipulate, and move tensors (CPU ↔ GPU) fluently
- [ ] Understand autograd: computational graph, `.backward()`, `.grad`
- [ ] Build custom `nn.Module` networks from scratch
- [ ] Write a complete, correct training loop (DataLoader → optimizer → loss → backward → step)
- [ ] Save and load model checkpoints; use device-agnostic code
- [ ] Use hooks, `named_parameters()`, and mixed precision (`torch.autocast`)
- [ ] Implement `nn.Embedding`, `nn.MultiheadAttention`, and a simple transformer block
- [ ] Debug shape mismatches, NaN gradients, and in-place operation errors

---

## Self-Check After Completing This Notebook
1. From memory: write a 3-layer MLP `nn.Module` with ReLU activations.
2. What is the difference between `tensor.detach()` and `tensor.data`? Why does it matter?
3. What does `.zero_grad()` do and why must it be called before `.backward()`?
4. How do you make code that runs on CPU and GPU without if/else branches everywhere?
5. Explain in one sentence what `torch.autocast` does and why it speeds up training.

## 🧬 PyTorch — Concepts for Beginners

| Term | Plain English |
|------|--------------|
| **Tensor** | PyTorch's multi-dimensional array (like NumPy ndarray but with GPU support and autograd) |
| **Gradient** | The derivative of a loss w.r.t. a parameter — tells the optimizer which direction to step |
| **Autograd** | PyTorch's automatic differentiation engine; builds a computation graph on the fly |
| **backward()** | The method that computes gradients by backpropagating through the computation graph |
| **Optimizer** | Algorithm that updates model parameters using gradients (SGD, Adam, AdamW) |
| **Loss function** | A scalar that measures how wrong the model's predictions are (MSE, CrossEntropy) |
| **dtype** | Data type of tensor elements (`torch.float32`, `torch.long`, etc.) |
| **device** | Where the tensor lives: `cpu`, `cuda:0` (NVIDIA GPU), or `mps` (Apple GPU) |
| **requires_grad** | Flag that tells PyTorch to track operations on this tensor for gradient computation |
| **.detach()** | Returns a tensor disconnected from the computation graph (no gradient tracking) |
| **view / reshape** | Change tensor shape without copying data (`view` needs contiguous memory) |
| **Broadcasting** | Automatic expansion of smaller tensors to match shapes during element-wise ops |

## Beginner Teaching Frame

**Who should fully work through this notebook:** students with little or no ML background.

**How to study it on a first pass:**
- read every markdown section before touching the code
- run the code in small chunks
- paraphrase each concept in your own words
- write one tiny example from memory after finishing

**Common traps:** memorizing syntax without understanding the data flow, skipping probability, and rushing past train/test split discipline.


## Canonical Resource Upgrade

If you need a second explanation, use these first:
- [CS50P](https://cs50.harvard.edu/python/) for programming fundamentals
- [Harvard Stat 110](https://stat110.hsites.harvard.edu/) for probability intuition
- [scikit-learn MOOC](https://inria.github.io/scikit-learn-mooc/) for correct ML workflow
- [PyTorch Tutorials](https://docs.pytorch.org/tutorials/) once you reach the deep learning notebooks


## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | 00/01 Python Core + 00/02 ML Fundamentals — numpy, calculus intuition, gradient descent concept |
| You Are Here | Module 00/06 — PyTorch Fundamentals |
| Next Steps | 05/01 Deep Learning & Fine-tuning, 06/01 Structural ML/GNNs, 07/01 AlphaFold3, 10/01 OpenFold3 Fine-tuning |
| Goal | Implement full PyTorch training loop; understand autograd; run ESM2 protein inference |

### From Scratch? Start Here:
1. [Karpathy Neural Networks Zero to Hero (free)](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ) — build backprop from scratch
2. [PyTorch 60-minute blitz (free)](https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html)
3. This notebook

**Time:** 12-20 hours | **Difficulty:** ⭐⭐⭐⭐

### Cross-References
- Builds on: 00/01 Python Core, 00/02 ML Fundamentals
- Used by: 05/01 Deep Learning, 06/01 Structural ML, 07/01-07/04 AlphaFold3, 10/01 OpenFold3

## 🤖 AI Walkthrough — Claude Prompts

Open a Claude conversation alongside this notebook and use these prompts:

**On tensors and autograd:**
```
Explain PyTorch's computational graph to me.
When I call loss.backward(), what is stored in the graph and how does PyTorch
know how to compute each gradient? Walk through a simple y = x*w + b example
step by step, showing what leaf vs non-leaf tensors are.
```

**On nn.Module:**
```
Explain how nn.Module works in PyTorch.
What does __init__ vs forward do? Where are parameters stored?
If I call model.parameters(), what exactly does it return and why does
the optimizer need it?
```

**On the training loop:**
```
Walk me through the exact sequence of a PyTorch training loop step:
1. zero_grad()
2. forward pass
3. loss computation
4. backward()
5. optimizer.step()
For each step: what state changes? What would break if I skipped it?
```

**On device management:**
```
Explain PyTorch device management. What happens when a tensor is on CPU
but the model is on GPU? Show me the idiomatic pattern for writing device-agnostic
code that works whether CUDA is available or not.
```

**On attention mechanisms:**
```
Explain nn.MultiheadAttention in PyTorch.
What are the q, k, v inputs? What does 'multihead' add vs single-head attention?
Why does AlphaFold and ESM2 use this as their core building block?
```

**On debugging:**
```
I'm getting 'RuntimeError: Expected all tensors to be on the same device'.
Walk me through the systematic debugging process. What are the 5 most common
causes of this error and how do I fix each one?
```

---

## Interview Tip Bank
> **Most common PyTorch interview question**: Explain the training loop. They want to hear: zero_grad → forward → loss → backward → step, and WHY each step is needed.

> **Autograd gotcha**: Gradients accumulate by default. Forgetting `.zero_grad()` is the #1 beginner bug — gradients from previous batches pile up.

> **eval() vs train()**: Always call `model.eval()` before inference and `model.train()` before training. BatchNorm and Dropout behave differently in each mode.

> **Leaf vs non-leaf tensors**: Only leaf tensors (inputs, parameters) with `requires_grad=True` accumulate `.grad`. Intermediate tensors store only the gradient function.

> **Mixed precision**: `torch.autocast` uses float16 for most ops, float32 for numerically sensitive ones. ~2x speedup, ~2x memory reduction on modern GPUs.

## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **ML fundamentals** — train/test splits, loss functions, gradient descent intuition. *Review: `00_python_ml_basics/02_ml_fundamentals.ipynb`*
- **NumPy arrays** — creating, reshaping, and indexing multi-dimensional arrays. *Review: `00_python_ml_basics/01_python_core_for_bioinformatics.ipynb`*

**Quick recap of terms used in this notebook:**
- **tensor** — a multi-dimensional array (like a numpy array but with GPU support and autograd)
- **gradient** — the derivative telling which direction to adjust weights to reduce loss
- **backpropagation** — computing gradients by applying the chain rule backwards through the network. *Full explanation in `05_deep_learning_finetuning/01_dl_and_finetuning.ipynb`*
- **learning rate** — step size for weight updates; too high = overshoot, too low = slow convergence


## Prerequisites

Before this notebook, you should be comfortable with:
- **Notebook 00/01**: Python basics (classes, functions, list comprehensions)
- **Notebook 00/01**: NumPy arrays (shapes, broadcasting, indexing)
- **Notebook 02**: Basic ML concepts (loss functions, gradient descent conceptually)

You do NOT need prior PyTorch experience. Everything is built from the ground up.

### Quick NumPy-to-PyTorch mapping (reference)
| NumPy | PyTorch | Notes |
|-------|---------|-------|
| `np.array([1,2,3])` | `torch.tensor([1,2,3])` | Creates from data |
| `np.zeros((3,4))` | `torch.zeros(3,4)` | Zeros tensor |
| `arr.shape` | `tensor.shape` | Same API |
| `arr @ arr2` | `tensor @ tensor2` | Matrix multiply |
| `arr.T` | `tensor.T` | Transpose |
| `arr.reshape(2,-1)` | `tensor.reshape(2,-1)` | Reshape |
| `arr.astype(np.float32)` | `tensor.float()` | Type cast |
| `tensor.numpy()` | `arr` | Back to NumPy (CPU only) |

### Install check
```bash
pip install torch torchvision  # CPU-only
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121  # CUDA 12.1
```

## Section 1 — Tensors & Autograd

### What is a Tensor?
A tensor is a multi-dimensional array — PyTorch's equivalent of NumPy's `ndarray`. The critical difference: tensors can **track gradient computation** and live on a **GPU**.

### What is Autograd?
Autograd is PyTorch's automatic differentiation engine. Every operation on a tensor with `requires_grad=True` is recorded in a **computation graph**. When you call `.backward()`, PyTorch traverses this graph in reverse (backpropagation) and populates `.grad` on all leaf tensors.

```
Forward pass:    x  →  w*x  →  loss
Backward pass:   dloss/dw  ←  (chain rule applied at each node)
```

### Key Concepts
- **Leaf tensor**: Created directly by user (model weights, inputs). These accumulate `.grad`.
- **Non-leaf tensor**: Result of an operation. Stores only the gradient function (`.grad_fn`).
- **`.detach()`**: Creates a new tensor sharing data but NOT in the computation graph.
- **`torch.no_grad()`**: Context manager that disables gradient tracking entirely (saves memory during inference).

In [ ]:
# PyTorch Tensors & Autograd
import torch
import torch.nn.functional as F

# Tensor creation
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(f"Tensor: {x}")
print(f"Shape: {x.shape}, dtype: {x.dtype}, device: {x.device}")

# Autograd: track operations for automatic differentiation
w = torch.randn(3, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# Forward pass: simple linear regression prediction
x0 = torch.tensor([1.0, 2.0, 3.0])
pred = x0 @ w + b
loss = (pred - 5.0)**2  # MSE vs target=5

# Backward pass: compute gradients
loss.backward()
print(f"\nAutograd demo:")
print(f"w.grad: {w.grad}")
print(f"b.grad: {b.grad}")
print(f"Gradient = d(loss)/dw = 2*(pred-5)*x0 = {(2*(pred-5.0)*x0).detach()}")

# Key: gradient flows BACKWARD through computational graph
print("\nComputational graph: x → @w → +b → pred → (pred-5)^2 → loss")

> 📋 **Expected output:** A tensor printed as `tensor([...])` with the values you specified. The `dtype` should be `torch.float32` (default) or `torch.int64` for integers.
> ✅ If you see tensor output, PyTorch is working correctly.

## 🔧 Common Errors & Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `RuntimeError: CUDA out of memory` | GPU RAM exceeded | Reduce batch size, use `torch.no_grad()` for inference, or call `torch.cuda.empty_cache()` |
| `RuntimeError: expected scalar type Float but found Long` | Integer tensor where float expected | Cast with `.float()` or `.to(torch.float32)` |
| `RuntimeError: mat1 and mat2 shapes cannot be multiplied` | Linear layer input size mismatch | Print `x.shape` before the layer; ensure `in_features` matches |
| `RuntimeError: element 0 of tensors does not require grad` | Calling `.backward()` on a detached tensor | Ensure `requires_grad=True` on the tensor or that it depends on a leaf with grad |
| `UserWarning: Using a target size that is different from the input size` | Shape mismatch in loss function | Squeeze or unsqueeze tensors to match (e.g., `target.unsqueeze(1)`) |
| `RuntimeError: Expected all tensors on the same device` | Mixing CPU and GPU tensors | Move both to the same device with `.to(device)` |

## 🎤 Interview Questions

**Q1 (Easy):** What is the difference between `torch.Tensor` and `numpy.ndarray`?
<details><summary>Answer</summary>
Both are multi-dimensional arrays, but PyTorch tensors support automatic differentiation (autograd), can live on GPU, and integrate with PyTorch's neural network modules. NumPy arrays are CPU-only and have no built-in gradient tracking. You can convert between them with `.numpy()` and `torch.from_numpy()` (they share memory on CPU).
</details>

**Q2 (Easy):** What does `model.eval()` do and why is it important during inference?
<details><summary>Answer</summary>
`model.eval()` switches layers like Dropout and BatchNorm to evaluation mode. Dropout stops dropping units, and BatchNorm uses running statistics instead of batch statistics. Without it, inference results are non-deterministic and degraded. Pair it with `torch.no_grad()` to also disable gradient tracking and save memory.
</details>

**Q3 (Medium):** Explain the PyTorch autograd computation graph. What happens when you call `.backward()`?
<details><summary>Answer</summary>
PyTorch dynamically builds a directed acyclic graph (DAG) where nodes are tensors and edges are operations. When you call `loss.backward()`, it traverses this graph in reverse (topological order), applying the chain rule to compute gradients for every leaf tensor with `requires_grad=True`. Gradients accumulate in `.grad` attributes. The graph is then destroyed (unless `retain_graph=True`).
</details>

**Q4 (Medium):** Why might you use `torch.no_grad()` vs `.detach()`? When would you choose one over the other?
<details><summary>Answer</summary>
`torch.no_grad()` is a context manager that disables gradient computation for all operations inside it — ideal for inference loops. `.detach()` disconnects a single tensor from the graph, returning a new tensor that shares data but has no grad history — useful when you need the tensor's value as a constant (e.g., target in a loss, or for logging). Use `no_grad()` for blocks of code, `.detach()` for individual tensors.
</details>

**Q5 (Hard):** How does PyTorch handle memory on GPU, and what strategies can you use to train models that do not fit in GPU memory?
<details><summary>Answer</summary>
PyTorch uses a caching memory allocator on GPU — freed tensors are kept in a cache for reuse, not returned to CUDA immediately. Strategies for large models: (1) gradient accumulation — run multiple forward passes and sum gradients before stepping; (2) mixed-precision training (`torch.cuda.amp`) — use float16 for forward pass, float32 for critical ops; (3) gradient checkpointing (`torch.utils.checkpoint`) — recompute activations during backward instead of storing them; (4) model parallelism — split layers across GPUs; (5) DeepSpeed/FSDP for fully sharded data parallelism.
</details>

## ✅ Notebook Complete

**You can now:**
- Create, manipulate, and move PyTorch tensors between CPU and GPU
- Build and train a simple neural network using `nn.Module`
- Use autograd to compute gradients and understand the computation graph
- Apply common tensor operations: reshape, broadcast, index, slice

**Next:** → `05_deep_learning_finetuning/01_dl_and_finetuning.ipynb`